# N2 — The Kalman Filter: Recursive State Estimation from Scratch

This notebook implements the Kalman filter from first principles — no estimation libraries, just NumPy matrices. We build intuition for the predict/update loop, watch the Kalman gain converge, tune noise parameters, and see the filter coast gracefully through sensor dropout.

**What you will learn:**
1. The recursive Bayes filter structure: predict widens belief, update narrows it
2. How to tune process noise **Q** and measurement noise **R** to balance responsiveness vs smoothness
3. The Kalman gain as a "trust knob" between prediction and measurement
4. Matrix-form predict/update for arbitrary linear systems
5. Coasting behavior — graceful degradation through measurement loss with calibrated uncertainty growth

**Pipeline overview:**

```
        PREDICT                          UPDATE
   (motion model)                   (measurement)

  x̂⁻ = F · x̂⁺                    K = P⁻ · Hᵀ · (H · P⁻ · Hᵀ + R)⁻¹
  P⁻ = F · P⁺ · Fᵀ + Q            x̂⁺ = x̂⁻ + K · (z - H · x̂⁻)
                                    P⁺ = (I - K · H) · P⁻
       │                                  │
       ▼                                  ▼
  Belief WIDENS                    Belief NARROWS
  (less certain)                   (more certain)
```

## 1. Install and Import Dependencies

In [ ]:
%pip install numpy matplotlib --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.gridspec as gridspec

np.random.seed(42)

print("All libraries loaded successfully!")

## The Recursive Bayes Filter

Every Kalman filter is a special case of the **recursive Bayes filter** — a two-step loop that maintains a probability distribution (the *belief*) over the system's hidden state:

**Predict** — use a motion model to propagate the belief forward in time. Because the model is imperfect, uncertainty *grows*:

$$\bar{\text{bel}}(x_t) = \int p(x_t \mid u_t, x_{t-1}) \; \text{bel}(x_{t-1}) \; dx_{t-1}$$

**Update** — incorporate a new measurement to *shrink* uncertainty:

$$\text{bel}(x_t) = \eta \; p(z_t \mid x_t) \; \bar{\text{bel}}(x_t)$$

When the motion model and measurement model are both **linear** and the noise is **Gaussian**, these integrals have a closed-form solution — the **Kalman filter**. The belief stays Gaussian forever, fully described by a mean vector $\hat{x}$ and covariance matrix $P$.

### Kalman Filter Equations

| Step | Equation | Meaning |
|------|----------|---------|
| **Predict state** | $\hat{x}^- = F \hat{x}^+$ | Propagate the mean through the motion model |
| **Predict covariance** | $P^- = F P^+ F^\top + Q$ | Propagate uncertainty; $Q$ adds process noise |
| **Kalman gain** | $K = P^- H^\top (H P^- H^\top + R)^{-1}$ | Balance trust between prediction and measurement |
| **Update state** | $\hat{x}^+ = \hat{x}^- + K(z - H\hat{x}^-)$ | Correct the prediction toward the measurement |
| **Update covariance** | $P^+ = (I - KH) P^-$ | Shrink uncertainty by the information gained |

where:
- $F$ — state transition matrix (motion model)
- $H$ — measurement matrix (what the sensor observes)
- $Q$ — process noise covariance (how much we distrust the model)
- $R$ — measurement noise covariance (how much we distrust the sensor)
- $z$ — the actual measurement

## 2. 1D Kalman Filter — Tracking a Moving Object

Let's start with the simplest possible case: tracking a single value (position) with a constant-velocity model. The state is $[\,x,\; \dot{x}\,]^\top$ — position and velocity — but we only measure position.

**State transition (constant velocity):**

$$F = \begin{bmatrix} 1 & \Delta t \\ 0 & 1 \end{bmatrix}, \qquad H = \begin{bmatrix} 1 & 0 \end{bmatrix}$$

The object moves at roughly constant velocity, but we add process noise to model small accelerations. The sensor reports position with Gaussian noise.

In [ ]:
def kalman_filter(measurements, F, H, Q, R, x0, P0):
    '''
    Run a linear Kalman filter on a sequence of measurements.

    Returns arrays of state estimates, covariances, Kalman gains,
    and the predicted (pre-update) state and covariance at each step.
    '''
    n = len(measurements)
    dim_x = F.shape[0]
    dim_z = H.shape[0]

    x_estimates = np.zeros((n, dim_x))
    P_estimates = np.zeros((n, dim_x, dim_x))
    K_history = np.zeros((n, dim_x, dim_z))
    x_predictions = np.zeros((n, dim_x))
    P_predictions = np.zeros((n, dim_x, dim_x))

    x = x0.copy()
    P = P0.copy()

    for k in range(n):
        # --- PREDICT ---
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q

        x_predictions[k] = x_pred
        P_predictions[k] = P_pred

        # --- UPDATE ---
        y = measurements[k] - H @ x_pred          # innovation (residual)
        S = H @ P_pred @ H.T + R                   # innovation covariance
        K = P_pred @ H.T @ np.linalg.inv(S)        # Kalman gain

        x = x_pred + K @ y
        P = (np.eye(dim_x) - K @ H) @ P_pred

        x_estimates[k] = x
        P_estimates[k] = P
        K_history[k] = K

    return x_estimates, P_estimates, K_history, x_predictions, P_predictions


# ╔══════════════════════════════════════════════════════════════╗
# ║  TUNABLE PARAMETERS — try changing these and re-running!    ║
# ╚══════════════════════════════════════════════════════════════╝

# Q: Process noise — how much do you trust the motion model?
#    Low  (e.g. 0.05) → filter trusts the model, smooth but slow to react
#    High (e.g. 5.0)  → filter distrusts the model, responsive but noisy
process_noise_std = 0.5       # try: 0.05, 0.5, 2.0, 5.0

# R: Measurement noise — how much do you trust the sensor?
#    Low  (e.g. 2.0)  → filter trusts measurements, tracks closely
#    High (e.g. 30.0) → filter distrusts measurements, smooths heavily
measurement_noise_std = 10.0  # try: 2.0, 10.0, 30.0

# ════════════════════════════════════════════════════════════════

# --- Generate synthetic data ---
dt = 1.0
num_steps = 80
true_velocity = 2.0

true_positions = np.array([true_velocity * dt * k for k in range(num_steps)])
measurements = true_positions + np.random.normal(0, measurement_noise_std, num_steps)

# --- Set up filter ---
F = np.array([[1, dt],
              [0, 1]])

H = np.array([[1.0, 0.0]])

Q = np.array([[dt**4/4, dt**3/2],
              [dt**3/2, dt**2  ]]) * process_noise_std**2

R = np.array([[measurement_noise_std**2]])

x0 = np.array([0.0, 0.0])
P0 = np.array([[500.0, 0.0],
               [0.0,  50.0]])

# --- Run filter ---
x_est, P_est, K_hist, x_pred, P_pred = kalman_filter(
    measurements.reshape(-1, 1), F, H, Q, R, x0, P0)

print(f"True velocity:            {true_velocity:.1f} m/s")
print(f"Process noise (Q scale):  σ_q = {process_noise_std:.2f}")
print(f"Measurement noise (R):    σ_r = {measurement_noise_std:.1f}")
print(f"Q/R ratio:                {process_noise_std**2 / measurement_noise_std**2:.4f}")
print(f"Final estimated velocity: {x_est[-1, 1]:.2f} m/s")
print(f"Final position error:     {abs(x_est[-1, 0] - true_positions[-1]):.2f} m")

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

time = np.arange(num_steps)
pos_std = np.sqrt(P_est[:, 0, 0])
vel_std = np.sqrt(P_est[:, 1, 1])

fig, axes = plt.subplots(2, 1, figsize=(14, 9), dpi=80,
                         gridspec_kw={"height_ratios": [3, 1]})

# --- Top: Position tracking ---
ax_pos = axes[0]
ax_pos.set_xlabel("Time step", fontsize=12)
ax_pos.set_ylabel("Position (m)", fontsize=12)
ax_pos.set_title("1D Kalman Filter — Constant-Velocity Tracking", fontsize=14)
ax_pos.grid(True, alpha=0.3)
ax_pos.set_xlim(-1, num_steps)
ax_pos.set_ylim(min(measurements.min(), true_positions.min()) - 15,
                max(measurements.max(), true_positions.max()) + 15)

true_ln, = ax_pos.plot([], [], "k-", linewidth=2, label="True position")
meas_sc = ax_pos.scatter([], [], c="red", s=15, alpha=0.4,
                          label="Noisy measurements", zorder=3)
est_ln, = ax_pos.plot([], [], "b-", linewidth=2, label="Kalman estimate")
ax_pos.legend(fontsize=11, loc="upper left")

# --- Bottom: Velocity estimate ---
ax_vel = axes[1]
ax_vel.axhline(true_velocity, color="k", linewidth=1.5, linestyle="--",
               label="True velocity")
ax_vel.set_xlabel("Time step", fontsize=12)
ax_vel.set_ylabel("Velocity (m/s)", fontsize=12)
ax_vel.grid(True, alpha=0.3)
ax_vel.set_xlim(-1, num_steps)
ax_vel.set_ylim(min(x_est[:, 1].min() - 2, -1),
                max(true_velocity + 2, x_est[:, 1].max() + 2))

vel_ln, = ax_vel.plot([], [], "b-", linewidth=2, label="Estimated velocity")
ax_vel.legend(fontsize=11)
plt.tight_layout()

pos_band = [None]
vel_band = [None]

def update_1d(frame):
    s = slice(0, frame + 1)
    t = time[s]

    true_ln.set_data(t, true_positions[s])
    meas_sc.set_offsets(np.column_stack([t, measurements[s]]))
    est_ln.set_data(t, x_est[s, 0])
    vel_ln.set_data(t, x_est[s, 1])

    if pos_band[0] is not None:
        pos_band[0].remove()
    pos_band[0] = ax_pos.fill_between(
        t, x_est[s, 0] - 2*pos_std[s], x_est[s, 0] + 2*pos_std[s],
        color="blue", alpha=0.15)

    if vel_band[0] is not None:
        vel_band[0].remove()
    vel_band[0] = ax_vel.fill_between(
        t, x_est[s, 1] - 2*vel_std[s], x_est[s, 1] + 2*vel_std[s],
        color="blue", alpha=0.15)

    return (true_ln, meas_sc, est_ln, vel_ln)

anim_1d = FuncAnimation(fig, update_1d, frames=num_steps, interval=60, blit=False)
plt.close(fig)

print(f"Animation: {num_steps} frames  |  Press ▶ to play")
display(HTML(anim_1d.to_jshtml()))

## 3. Kalman Gain Convergence

The Kalman gain $K$ starts large (the filter trusts measurements heavily because its initial state estimate is uncertain) and converges to a steady-state value as the filter becomes more confident. This convergence depends only on $F$, $H$, $Q$, and $R$ — not the actual measurements.

- **High gain** → filter tracks measurements closely (responsive but noisy)
- **Low gain** → filter trusts its model more (smooth but slow to react)

In [ ]:
pred_std = np.sqrt(P_pred[:, 0, 0])
post_std = np.sqrt(P_est[:, 0, 0])

fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=80)

# Left: Kalman gain
ax_gain = axes[0]
ax_gain.set_xlabel("Time step", fontsize=12)
ax_gain.set_ylabel("Kalman Gain", fontsize=12)
ax_gain.set_title("Kalman Gain Convergence", fontsize=14)
ax_gain.grid(True, alpha=0.3)
ax_gain.set_xlim(-1, num_steps)
ax_gain.set_ylim(0, max(K_hist[:, 0, 0].max(), K_hist[:, 1, 0].max()) * 1.15)

k_pos_ln, = ax_gain.plot([], [], "b-", linewidth=2, label="K (position)")
k_vel_ln, = ax_gain.plot([], [], "r-", linewidth=2, label="K (velocity)")
ax_gain.legend(fontsize=11)

# Right: uncertainty oscillation
ax_unc = axes[1]
ax_unc.set_xlabel("Time step", fontsize=12)
ax_unc.set_ylabel("Position Std Dev (m)", fontsize=12)
ax_unc.set_title("Uncertainty: Predict Widens, Update Narrows", fontsize=14)
ax_unc.grid(True, alpha=0.3)
ax_unc.set_xlim(-1, num_steps)
ax_unc.set_ylim(0, max(pred_std.max(), post_std.max()) * 1.1)

pred_ln, = ax_unc.plot([], [], "r--", linewidth=2, label="Predicted σ (before update)")
post_ln, = ax_unc.plot([], [], "b-", linewidth=2, label="Posterior σ (after update)")
ax_unc.legend(fontsize=11)
plt.tight_layout()

def update_gain(frame):
    s = slice(0, frame + 1)
    t = time[s]

    k_pos_ln.set_data(t, K_hist[s, 0, 0])
    k_vel_ln.set_data(t, K_hist[s, 1, 0])
    pred_ln.set_data(t, pred_std[s])
    post_ln.set_data(t, post_std[s])

    return (k_pos_ln, k_vel_ln, pred_ln, post_ln)

anim_gain = FuncAnimation(fig, update_gain, frames=num_steps, interval=60, blit=False)
plt.close(fig)

print(f"Animation: {num_steps} frames  |  Press ▶ to play")
print(f"Watch the gain converge: {K_hist[0, 0, 0]:.3f} → {K_hist[-1, 0, 0]:.3f}")
display(HTML(anim_gain.to_jshtml()))

## Understanding Q and R as Trust Knobs

The ratio $Q / R$ controls the filter's personality:

| Setting | Q/R Ratio | Behavior | Analogy |
|---------|-----------|----------|---------|
| **Stiff** | Low Q, high R | Trusts the model; smooths aggressively | "I know the physics, the sensor is noisy" |
| **Balanced** | Moderate | Tracks the signal without excess noise | The sweet spot |
| **Floppy** | High Q, low R | Trusts measurements; tracks every wiggle | "The sensor is gospel, the model is approximate" |

There is no universally correct setting — it depends on how well your motion model matches reality and how accurate your sensor is. The art of Kalman filtering is choosing $Q$ and $R$ to match the actual noise characteristics of your system.

## 4. Q vs R Tuning: Stiff, Balanced, and Floppy

Let's run three filters on the same noisy data with different process-noise scales and compare their tracking behavior on a sinusoidal trajectory where the constant-velocity model is imperfect.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TUNABLE PARAMETERS — try changing these and re-running!    ║
# ╚══════════════════════════════════════════════════════════════╝

# Measurement noise for all three filters (shared sensor)
meas_noise_std_tune = 8.0      # try: 3.0, 8.0, 20.0

# Q scales for each filter — controls how much each trusts its model
# Lower = stiffer (trusts model), Higher = floppier (trusts measurements)
q_stiff    = 0.01              # try: 0.001, 0.01, 0.1
q_balanced = 0.5               # try: 0.2, 0.5, 1.0
q_floppy   = 5.0               # try: 2.0, 5.0, 20.0

# ════════════════════════════════════════════════════════════════

# Sinusoidal + linear drift — the CV model is a poor fit, so Q matters
np.random.seed(42)
num_steps_tune = 120
dt_tune = 1.0
time_tune = np.arange(num_steps_tune)

true_pos_tune = 0.5 * time_tune + 15 * np.sin(0.08 * time_tune)
measurements_tune = true_pos_tune + np.random.normal(0, meas_noise_std_tune, num_steps_tune)

F_tune = np.array([[1, dt_tune], [0, 1]])
H_tune = np.array([[1.0, 0.0]])
R_tune = np.array([[meas_noise_std_tune**2]])

configs = [
    (f"Stiff (Q scale = {q_stiff})",     q_stiff),
    (f"Balanced (Q scale = {q_balanced})",  q_balanced),
    (f"Floppy (Q scale = {q_floppy})",    q_floppy),
]

# Run all 3 filters upfront and store results
results_tune = []
for label, q_scale in configs:
    Q_tune = np.array([[dt_tune**4/4, dt_tune**3/2],
                       [dt_tune**3/2, dt_tune**2  ]]) * q_scale**2
    x0_t = np.array([0.0, 0.0])
    P0_t = np.array([[500.0, 0.0], [0.0, 50.0]])
    x_est_t, P_est_t, _, _, _ = kalman_filter(
        measurements_tune.reshape(-1, 1), F_tune, H_tune, Q_tune, R_tune, x0_t, P0_t)
    pos_std_t = np.sqrt(P_est_t[:, 0, 0])
    rmse = np.sqrt(np.mean((x_est_t[:, 0] - true_pos_tune)**2))
    results_tune.append((label, x_est_t, pos_std_t, rmse))

# --- Animate all 3 configs side-by-side ---
step_t = 2
frame_indices_t = list(range(0, num_steps_tune, step_t))

fig, axes_t = plt.subplots(3, 1, figsize=(16, 12), dpi=80, sharex=True)

artists_t = []
bands_t = [None, None, None]

ymin_t = min(true_pos_tune.min(), measurements_tune.min()) - 12
ymax_t = max(true_pos_tune.max(), measurements_tune.max()) + 12

for i, (ax, (label, x_est_t, pos_std_t, rmse)) in enumerate(zip(axes_t, results_tune)):
    ax.set_ylabel("Position (m)", fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-1, num_steps_tune)
    ax.set_ylim(ymin_t, ymax_t)
    ax.set_title(f"{label}  —  RMSE = {rmse:.2f} m", fontsize=13)

    true_l, = ax.plot([], [], "k-", linewidth=2, label="True")
    meas_s = ax.scatter([], [], c="red", s=10, alpha=0.3, label="Measurements")
    est_l, = ax.plot([], [], "b-", linewidth=2, label="Estimate")
    ax.legend(fontsize=10, loc="upper left")

    artists_t.append((true_l, meas_s, est_l, x_est_t, pos_std_t))

axes_t[-1].set_xlabel("Time step", fontsize=12)
plt.tight_layout()

def update_tune(frame):
    k = frame_indices_t[frame]
    s = slice(0, k + 1)
    t = time_tune[s]

    for i, (true_l, meas_s, est_l, x_est_t, pos_std_t) in enumerate(artists_t):
        true_l.set_data(t, true_pos_tune[s])
        meas_s.set_offsets(np.column_stack([t, measurements_tune[s]]))
        est_l.set_data(t, x_est_t[s, 0])

        if bands_t[i] is not None:
            bands_t[i].remove()
        bands_t[i] = axes_t[i].fill_between(
            t, x_est_t[s, 0] - 2*pos_std_t[s], x_est_t[s, 0] + 2*pos_std_t[s],
            color="blue", alpha=0.12)

    return tuple(item for tup in artists_t for item in tup[:3])

anim_tune = FuncAnimation(fig, update_tune, frames=len(frame_indices_t),
                          interval=50, blit=False)
plt.close(fig)

print(f"Animation: {len(frame_indices_t)} frames  |  Press ▶ to play")
print(f"R (measurement noise): σ_r = {meas_noise_std_tune}")
print(f"Q scales: stiff = {q_stiff}, balanced = {q_balanced}, floppy = {q_floppy}")
print("Compare: stiff lags behind curves, floppy tracks noise, balanced is the sweet spot")
display(HTML(anim_tune.to_jshtml()))

### Interactive Explorer: Drag Q and R

The stiff/balanced/floppy comparison above used fixed settings. Now it's your turn — use the sliders below to sweep Q and R continuously and build intuition for how the filter responds.

**The measurements stay fixed** (generated with the same noise as the cell above). The sliders only change the filter's internal Q and R — so you're exploring what happens when the filter's assumptions match or mis-match reality:

- **Crank Q up** — the filter admits the model is poor, trusts measurements more, Kalman gain rises
- **Crank R up past the true noise** — the filter thinks the sensor is worse than it is, over-smooths, lags behind curves
- **Crank R down below the true noise** — the filter over-trusts noisy measurements, tracks every wiggle
- **Set R ≈ 8 (true noise)** — the filter's belief matches reality, RMSE is minimized
- **Watch the summary panel** — it tells you whether R is well-calibrated, over-smoothing, or over-trusting

In [ ]:
%pip install ipywidgets --quiet

import ipywidgets as widgets

# Measurements are FIXED — generated once with the same noise as the animation above.
# The sliders only change the filter's Q and R, not the data it sees.
# This lets students explore correct tuning vs mis-specified parameters.

# Precompute fixed axis limits from the data so plots never rescale.
_pos_ymin = min(true_pos_tune.min(), measurements_tune.min()) - 15
_pos_ymax = max(true_pos_tune.max(), measurements_tune.max()) + 15
_true_vel = np.gradient(true_pos_tune, dt_tune)
_vel_ymin = _true_vel.min() - 3
_vel_ymax = _true_vel.max() + 3

def _run_interactive_kf(q_scale, r_scale):
    Q_i = np.array([[dt_tune**4/4, dt_tune**3/2],
                    [dt_tune**3/2, dt_tune**2  ]]) * q_scale**2
    R_i = np.array([[r_scale**2]])

    x_est_i, P_est_i, K_hist_i, _, _ = kalman_filter(
        measurements_tune.reshape(-1, 1), F_tune, H_tune, Q_i, R_i,
        np.array([0.0, 0.0]), np.array([[500.0, 0.0], [0.0, 50.0]]))

    pos_std_i = np.sqrt(P_est_i[:, 0, 0])
    vel_std_i = np.sqrt(P_est_i[:, 1, 1])
    rmse = np.sqrt(np.mean((x_est_i[:, 0] - true_pos_tune)**2))
    qr_ratio = q_scale**2 / r_scale**2

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))

    # Top left: position tracking
    ax = axes[0, 0]
    ax.plot(time_tune, true_pos_tune, 'k-', linewidth=2, label='True')
    ax.scatter(time_tune, measurements_tune, c='red', s=10, alpha=0.3,
               label=f'Measurements (true noise = {meas_noise_std_tune})')
    ax.plot(time_tune, x_est_i[:, 0], 'b-', linewidth=2, label='Kalman estimate')
    ax.fill_between(time_tune, x_est_i[:, 0] - 2*pos_std_i,
                    x_est_i[:, 0] + 2*pos_std_i, color='blue', alpha=0.12)
    ax.set_xlim(-1, num_steps_tune)
    ax.set_ylim(_pos_ymin, _pos_ymax)
    ax.set_ylabel('Position (m)', fontsize=11)
    ax.set_title(f'Position Tracking  |  RMSE = {rmse:.2f} m', fontsize=13)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

    # Top right: velocity estimate
    ax = axes[0, 1]
    ax.plot(time_tune, _true_vel, 'k-', linewidth=2, label='True velocity')
    ax.plot(time_tune, x_est_i[:, 1], 'b-', linewidth=2, label='Estimated velocity')
    ax.fill_between(time_tune, x_est_i[:, 1] - 2*vel_std_i,
                    x_est_i[:, 1] + 2*vel_std_i, color='blue', alpha=0.12)
    ax.set_xlim(-1, num_steps_tune)
    ax.set_ylim(_vel_ymin, _vel_ymax)
    ax.set_ylabel('Velocity (m/s)', fontsize=11)
    ax.set_title('Velocity Estimate', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Bottom left: Kalman gain
    ax = axes[1, 0]
    ax.plot(time_tune, K_hist_i[:, 0, 0], 'b-', linewidth=2, label='K (position)')
    ax.plot(time_tune, K_hist_i[:, 1, 0], 'r-', linewidth=2, label='K (velocity)')
    ax.set_xlim(-1, num_steps_tune)
    ax.set_ylim(0, 1.2)
    ax.set_xlabel('Time step', fontsize=11)
    ax.set_ylabel('Kalman Gain', fontsize=11)
    ax.set_title(f'Kalman Gain  |  steady-state K_pos = {K_hist_i[-1,0,0]:.3f}', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Bottom right: summary panel
    ax = axes[1, 1]
    ax.axis('off')
    if qr_ratio < 0.005:
        personality = "STIFF  (trusts model heavily)"
        bg = '#d4e6f1'
    elif qr_ratio > 0.1:
        personality = "FLOPPY  (trusts sensor heavily)"
        bg = '#fadbd8'
    else:
        personality = "BALANCED"
        bg = '#d5f5e3'

    r_match = abs(r_scale - meas_noise_std_tune) / meas_noise_std_tune
    if r_match < 0.15:
        r_status = f"R ~ true noise  (well-calibrated)"
    elif r_scale > meas_noise_std_tune:
        r_status = f"R > true noise  (over-smoothing)"
    else:
        r_status = f"R < true noise  (over-trusting sensor)"

    info = (
        f"Filter's Q   sigma_q = {q_scale:.4f}\n"
        f"Filter's R   sigma_r = {r_scale:.2f}\n"
        f"True noise   sigma   = {meas_noise_std_tune:.2f}\n\n"
        f"Q/R ratio = {qr_ratio:.6f}\n\n"
        f"Position RMSE = {rmse:.2f} m\n"
        f"Noise reduction = {meas_noise_std_tune / max(rmse, 0.01):.1f}x\n\n"
        f"Steady-state gains:\n"
        f"  K_pos = {K_hist_i[-1,0,0]:.4f}\n"
        f"  K_vel = {K_hist_i[-1,1,0]:.4f}\n\n"
        f"R calibration: {r_status}\n"
        f"Personality:   {personality}"
    )
    ax.text(0.05, 0.95, info, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.5', facecolor=bg, alpha=0.8))
    ax.set_title('Filter Summary', fontsize=13)

    plt.tight_layout()
    plt.show()


q_slider = widgets.FloatLogSlider(
    value=0.5, base=10, min=-2, max=1.5, step=0.05,
    description='Q (process noise):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='550px'),
    readout_format='.4f'
)

r_slider = widgets.FloatLogSlider(
    value=8.0, base=10, min=0.3, max=1.7, step=0.05,
    description='R (meas. noise):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='550px'),
    readout_format='.2f'
)

out = widgets.interactive_output(_run_interactive_kf, {'q_scale': q_slider, 'r_scale': r_slider})

display(widgets.VBox([
    widgets.HTML(
        "<h4>Interactive Q / R Tuning</h4>"
        "<p style='color: gray; margin-top: -8px;'>"
        "The measurements are fixed (true sensor noise = "
        f"{meas_noise_std_tune:.1f}). The sliders only change the "
        "filter's Q and R parameters — so you can explore what happens "
        "when R matches reality vs. when it's mis-specified.</p>"
    ),
    q_slider, r_slider, out
]))

## 5. 2D Constant-Velocity Tracking (Full Matrix Form)

Now we generalize to 2D: tracking an object moving on a plane. The state vector is:

$$\hat{x} = \begin{bmatrix} x \\ y \\ \dot{x} \\ \dot{y} \end{bmatrix}$$

**State transition** (constant velocity in 2D):

$$F = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

**Measurement** — the sensor observes position only (not velocity):

$$H = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix}$$

This is the same filter structure that will appear in N4 (Multi-Object Tracker), where each YOLO detection's bounding-box center is tracked independently.

In [ ]:
class KalmanFilter2D:
    '''
    General-purpose linear Kalman filter in matrix form.
    No library shortcuts -- just NumPy.
    '''
    def __init__(self, F, H, Q, R, x0, P0):
        self.F = F
        self.H = H
        self.Q = Q
        self.R = R
        self.x = x0.copy()
        self.P = P0.copy()
        self.dim_x = F.shape[0]

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x.copy(), self.P.copy()

    def update(self, z):
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        I = np.eye(self.dim_x)
        self.P = (I - K @ self.H) @ self.P
        return self.x.copy(), self.P.copy(), K.copy()


# ╔══════════════════════════════════════════════════════════════╗
# ║  TUNABLE PARAMETERS — try changing these and re-running!    ║
# ╚══════════════════════════════════════════════════════════════╝

# Q: Process noise scale — how much unmodeled acceleration do you expect?
#    Low  (e.g. 0.05) → smooth trajectory, slow to react to turns
#    High (e.g. 3.0)  → responsive to maneuvers, but noisier
q_2d = 0.3                    # try: 0.05, 0.3, 1.0, 3.0

# R: Measurement noise — how noisy is the position sensor?
#    Low  (e.g. 1.0)  → filter trusts measurements closely
#    High (e.g. 15.0) → filter smooths heavily, ignoring outliers
meas_noise_2d = 5.0           # try: 1.0, 5.0, 15.0

# ════════════════════════════════════════════════════════════════

# --- Generate a 2D figure-eight trajectory ---
np.random.seed(42)
num_steps_2d = 200
dt_2d = 0.5
t_2d = np.arange(num_steps_2d) * dt_2d

true_x = 40 * np.sin(0.04 * t_2d)
true_y = 20 * np.sin(0.08 * t_2d)
true_vx = np.gradient(true_x, dt_2d)
true_vy = np.gradient(true_y, dt_2d)

meas_x = true_x + np.random.normal(0, meas_noise_2d, num_steps_2d)
meas_y = true_y + np.random.normal(0, meas_noise_2d, num_steps_2d)

# --- Set up 4-state KF ---
F_2d = np.array([[1, 0, dt_2d, 0    ],
                 [0, 1, 0,     dt_2d],
                 [0, 0, 1,     0    ],
                 [0, 0, 0,     1    ]])

H_2d = np.array([[1, 0, 0, 0],
                 [0, 1, 0, 0]], dtype=float)

Q_2d = q_2d**2 * np.array([
    [dt_2d**4/4, 0,          dt_2d**3/2, 0         ],
    [0,          dt_2d**4/4, 0,          dt_2d**3/2],
    [dt_2d**3/2, 0,          dt_2d**2,   0         ],
    [0,          dt_2d**3/2, 0,          dt_2d**2  ]])

R_2d = meas_noise_2d**2 * np.eye(2)

x0_2d = np.array([meas_x[0], meas_y[0], 0.0, 0.0])
P0_2d = np.diag([100.0, 100.0, 25.0, 25.0])

kf = KalmanFilter2D(F_2d, H_2d, Q_2d, R_2d, x0_2d, P0_2d)

# --- Run filter ---
estimates_2d = np.zeros((num_steps_2d, 4))
covariances_2d = np.zeros((num_steps_2d, 4, 4))

for k in range(num_steps_2d):
    kf.predict()
    z = np.array([meas_x[k], meas_y[k]])
    x_post, P_post, _ = kf.update(z)
    estimates_2d[k] = x_post
    covariances_2d[k] = P_post

rmse_x = np.sqrt(np.mean((estimates_2d[:, 0] - true_x)**2))
rmse_y = np.sqrt(np.mean((estimates_2d[:, 1] - true_y)**2))
rmse_total = np.sqrt(rmse_x**2 + rmse_y**2)

print(f"2D tracking RMSE:  x = {rmse_x:.2f} m,  y = {rmse_y:.2f} m,  total = {rmse_total:.2f} m")
print(f"Process noise:     σ_q = {q_2d}")
print(f"Measurement noise: σ_r = {meas_noise_2d:.1f} m")
print(f"Q/R ratio:         {q_2d**2 / meas_noise_2d**2:.4f}")
print(f"Noise reduction:   {meas_noise_2d / rmse_total:.1f}× improvement over raw measurements")

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Precompute error metrics for right panel
pos_error = np.sqrt((estimates_2d[:, 0] - true_x)**2 + (estimates_2d[:, 1] - true_y)**2)
meas_error = np.sqrt((meas_x - true_x)**2 + (meas_y - true_y)**2)
pos_sigma = np.sqrt(covariances_2d[:, 0, 0] + covariances_2d[:, 1, 1])

# Animate every 2nd step -> ~100 frames
step = 2
frame_indices = list(range(0, num_steps_2d, step))

fig, (ax_traj, ax_err) = plt.subplots(1, 2, figsize=(16, 7), dpi=80)

# --- Left: trajectory ---
ax_traj.plot(true_x, true_y, "k-", linewidth=1, alpha=0.15)
ax_traj.set_xlabel("X (m)", fontsize=12)
ax_traj.set_ylabel("Y (m)", fontsize=12)
ax_traj.set_title("2D Kalman Filter — Figure-Eight Tracking", fontsize=14)
ax_traj.set_aspect("equal")
ax_traj.grid(True, alpha=0.3)
margin = 12
ax_traj.set_xlim(true_x.min() - margin, true_x.max() + margin)
ax_traj.set_ylim(true_y.min() - margin, true_y.max() + margin)

meas_scat = ax_traj.scatter([], [], c="red", s=12, alpha=0.3, label="Measurements")
true_trail, = ax_traj.plot([], [], "k-", linewidth=2, label="True trajectory")
est_trail, = ax_traj.plot([], [], "b-", linewidth=2, label="Kalman estimate")
true_dot, = ax_traj.plot([], [], "ko", markersize=10, zorder=5)
est_dot, = ax_traj.plot([], [], "bs", markersize=8, zorder=5)

ell_anim = Ellipse((0, 0), 1, 1, fill=False, edgecolor="blue", linewidth=1.5, alpha=0.6)
ax_traj.add_patch(ell_anim)
ell_anim.set_visible(False)
ax_traj.legend(fontsize=10, loc="upper left")

# --- Right: error over time ---
ax_err.set_xlabel("Time (s)", fontsize=12)
ax_err.set_ylabel("Position Error (m)", fontsize=12)
ax_err.set_title("Error: Measurements vs Kalman Estimate", fontsize=14)
ax_err.grid(True, alpha=0.3)
ax_err.set_xlim(t_2d[0], t_2d[-1])
ax_err.set_ylim(0, max(meas_error.max(), pos_error.max()) * 1.1)

meas_err_line, = ax_err.plot([], [], "r-", alpha=0.3, linewidth=1, label="Measurement error")
est_err_line, = ax_err.plot([], [], "b-", linewidth=2, label="Estimate error")
sigma_line, = ax_err.plot([], [], "b--", linewidth=1.5, alpha=0.5, label="2σ bound")
ax_err.legend(fontsize=11)
plt.tight_layout()

def update_fig8(frame):
    k = frame_indices[frame]
    s = slice(0, k + 1)

    meas_scat.set_offsets(np.column_stack([meas_x[s], meas_y[s]]))
    true_trail.set_data(true_x[s], true_y[s])
    est_trail.set_data(estimates_2d[s, 0], estimates_2d[s, 1])
    true_dot.set_data([true_x[k]], [true_y[k]])
    est_dot.set_data([estimates_2d[k, 0]], [estimates_2d[k, 1]])

    cov_xy = covariances_2d[k, :2, :2]
    evals, evecs = np.linalg.eigh(cov_xy)
    angle = np.degrees(np.arctan2(evecs[1, 0], evecs[0, 0]))
    w, h = 2 * 2 * np.sqrt(evals)
    ell_anim.set_center((estimates_2d[k, 0], estimates_2d[k, 1]))
    ell_anim.width = w
    ell_anim.height = h
    ell_anim.angle = angle
    ell_anim.set_visible(True)

    meas_err_line.set_data(t_2d[s], meas_error[s])
    est_err_line.set_data(t_2d[s], pos_error[s])
    sigma_line.set_data(t_2d[s], 2 * pos_sigma[s])

    return (meas_scat, true_trail, est_trail, true_dot, est_dot,
            ell_anim, meas_err_line, est_err_line, sigma_line)

anim = FuncAnimation(fig, update_fig8, frames=len(frame_indices), interval=60, blit=False)
plt.close(fig)

print(f"Animation: {len(frame_indices)} frames  |  Press ▶ to play")
display(HTML(anim.to_jshtml()))

## 6. Sensor Dropout: Coasting Through Measurement Loss

What happens when the sensor goes dark? In the real world, measurements are lost all the time — occlusion, sensor failure, dropped packets. A well-tuned Kalman filter handles this gracefully by **coasting** on its motion model alone.

During a dropout:
- The **predict** step still runs (the filter's best guess propagates forward)
- The **update** step is skipped (no measurement available)
- **Uncertainty grows** monotonically — the filter knows it's becoming less certain
- When measurements resume, the filter **snaps back** — a large Kalman gain corrects the accumulated drift

This is exactly how the multi-object tracker in N4 keeps track of vehicles that are briefly occluded.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TUNABLE PARAMETERS — try changing these and re-running!    ║
# ╚══════════════════════════════════════════════════════════════╝

# Dropout window — when does the sensor go dark, and for how long?
#    Shorter gaps → filter barely notices
#    Longer gaps  → uncertainty balloons, drift accumulates
dropout_start = 70             # try: 40, 70, 100
dropout_end   = 130            # try: 90, 130, 180

# Q and R are inherited from the 2D filter cell above (q_2d, meas_noise_2d).
# Change them there to see how tuning affects coasting behavior.

# ════════════════════════════════════════════════════════════════

kf_drop = KalmanFilter2D(F_2d, H_2d, Q_2d, R_2d,
                          np.array([meas_x[0], meas_y[0], 0.0, 0.0]),
                          np.diag([100.0, 100.0, 25.0, 25.0]))

est_drop = np.zeros((num_steps_2d, 4))
cov_drop = np.zeros((num_steps_2d, 4, 4))
gains_drop = np.zeros(num_steps_2d)

for k in range(num_steps_2d):
    kf_drop.predict()

    if dropout_start <= k < dropout_end:
        # No measurement — coast on prediction only
        est_drop[k] = kf_drop.x.copy()
        cov_drop[k] = kf_drop.P.copy()
        gains_drop[k] = np.nan
    else:
        z = np.array([meas_x[k], meas_y[k]])
        x_post, P_post, K = kf_drop.update(z)
        est_drop[k] = x_post
        cov_drop[k] = P_post
        gains_drop[k] = np.linalg.norm(K)

# Precompute for visualization
err_drop = np.sqrt((est_drop[:, 0] - true_x)**2 + (est_drop[:, 1] - true_y)**2)
pos_sigma_drop = np.sqrt(cov_drop[:, 0, 0] + cov_drop[:, 1, 1])

dropout_len = (dropout_end - dropout_start) * dt_2d
print(f"Dropout window: steps {dropout_start}–{dropout_end}  "
      f"({dropout_len:.0f} s blackout)")
print(f"Filter settings: σ_q = {q_2d}, σ_r = {meas_noise_2d}")
print(f"Peak error during dropout:    {err_drop[dropout_start:dropout_end].max():.1f} m")
recovery_idx = min(dropout_end + 5, num_steps_2d - 1)
print(f"Error 5 steps after recovery: {err_drop[recovery_idx]:.1f} m")
print(f"Peak uncertainty (σ):         {pos_sigma_drop[dropout_start:dropout_end].max():.1f} m")

In [ ]:
step = 2
frame_indices_d = list(range(0, num_steps_2d, step))

fig, (ax_traj, ax_sig) = plt.subplots(1, 2, figsize=(16, 7), dpi=80)

# --- Left: trajectory ---
ax_traj.plot(true_x, true_y, "k-", linewidth=1, alpha=0.15)
ax_traj.set_xlabel("X (m)", fontsize=12)
ax_traj.set_ylabel("Y (m)", fontsize=12)
ax_traj.set_title("Sensor Dropout — 60-Step Blackout", fontsize=14)
ax_traj.set_aspect("equal")
ax_traj.grid(True, alpha=0.3)
all_x = np.concatenate([true_x, est_drop[:, 0], meas_x])
all_y = np.concatenate([true_y, est_drop[:, 1], meas_y])
pad = 10
ax_traj.set_xlim(all_x.min() - pad, all_x.max() + pad)
ax_traj.set_ylim(all_y.min() - pad, all_y.max() + pad)

meas_scat_d = ax_traj.scatter([], [], c="red", s=12, alpha=0.3, label="Measurements")
true_trail_d, = ax_traj.plot([], [], "k-", linewidth=2, label="True trajectory")
pre_line, = ax_traj.plot([], [], "b-", linewidth=2, label="Tracking")
coast_line, = ax_traj.plot([], [], color="orange", linewidth=2.5, linestyle="--",
                           label="Coasting (no measurements)")
post_line, = ax_traj.plot([], [], "g-", linewidth=2, label="Recovery")
true_dot_d, = ax_traj.plot([], [], "ko", markersize=10, zorder=5)
est_dot_d, = ax_traj.plot([], [], "s", color="blue", markersize=8, zorder=5)

ell_drop = Ellipse((0, 0), 1, 1, fill=False, edgecolor="blue", linewidth=1.5, alpha=0.6)
ax_traj.add_patch(ell_drop)
ell_drop.set_visible(False)

status_text = ax_traj.text(0.02, 0.98, "", transform=ax_traj.transAxes,
                           fontsize=12, fontweight="bold", va="top",
                           bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax_traj.legend(fontsize=9, loc="upper right")

# --- Right: uncertainty + error ---
ax_sig.set_xlabel("Time (s)", fontsize=12)
ax_sig.set_ylabel("(m)", fontsize=12)
ax_sig.set_title("Uncertainty & Error Over Time", fontsize=14)
ax_sig.grid(True, alpha=0.3)
ax_sig.set_xlim(t_2d[0], t_2d[-1])
ax_sig.set_ylim(0, max(pos_sigma_drop.max(), err_drop.max()) * 1.1)
ax_sig.axvspan(t_2d[dropout_start], t_2d[dropout_end - 1],
               color="orange", alpha=0.12, label="Dropout window")

sig_trail, = ax_sig.plot([], [], "b-", linewidth=2, label="Position σ")
err_trail, = ax_sig.plot([], [], "r-", linewidth=1.5, alpha=0.6, label="Position error")
time_marker = ax_sig.axvline(0, color="gray", linewidth=1, alpha=0.4, linestyle="--")
ax_sig.legend(fontsize=10)
plt.tight_layout()

def update_dropout(frame):
    k = frame_indices_d[frame]
    s = slice(0, k + 1)

    # True trajectory
    true_trail_d.set_data(true_x[s], true_y[s])
    true_dot_d.set_data([true_x[k]], [true_y[k]])

    # Measurements: visible only outside the dropout window
    vis = np.arange(k + 1)
    vis = vis[(vis < dropout_start) | (vis >= dropout_end)]
    if len(vis) > 0:
        meas_scat_d.set_offsets(np.column_stack([meas_x[vis], meas_y[vis]]))
    else:
        meas_scat_d.set_offsets(np.empty((0, 2)))

    # Estimate lines by phase
    pre_end = min(k + 1, dropout_start)
    pre_line.set_data(est_drop[:pre_end, 0], est_drop[:pre_end, 1])

    if k >= dropout_start:
        coast_end = min(k + 1, dropout_end)
        coast_line.set_data(est_drop[dropout_start:coast_end, 0],
                           est_drop[dropout_start:coast_end, 1])
    else:
        coast_line.set_data([], [])

    if k >= dropout_end:
        post_line.set_data(est_drop[dropout_end:k+1, 0],
                          est_drop[dropout_end:k+1, 1])
    else:
        post_line.set_data([], [])

    est_dot_d.set_data([est_drop[k, 0]], [est_drop[k, 1]])
    if dropout_start <= k < dropout_end:
        est_dot_d.set_color("orange")
    elif k >= dropout_end:
        est_dot_d.set_color("green")
    else:
        est_dot_d.set_color("blue")

    # Confidence ellipse — grows during dropout, shrinks on recovery
    cov_xy = cov_drop[k, :2, :2]
    evals, evecs = np.linalg.eigh(cov_xy)
    angle = np.degrees(np.arctan2(evecs[1, 0], evecs[0, 0]))
    w, h = 2 * 2 * np.sqrt(evals)
    ell_drop.set_center((est_drop[k, 0], est_drop[k, 1]))
    ell_drop.width = w
    ell_drop.height = h
    ell_drop.angle = angle
    ell_drop.set_visible(True)
    if dropout_start <= k < dropout_end:
        ell_drop.set_edgecolor("orange")
        ell_drop.set_linestyle("--")
    elif k >= dropout_end:
        ell_drop.set_edgecolor("green")
        ell_drop.set_linestyle("-")
    else:
        ell_drop.set_edgecolor("blue")
        ell_drop.set_linestyle("-")

    # Status banner
    if k < dropout_start:
        status_text.set_text("TRACKING")
        status_text.get_bbox_patch().set_facecolor("lightblue")
    elif k < dropout_end:
        status_text.set_text("SENSOR DROPOUT — coasting on model")
        status_text.get_bbox_patch().set_facecolor("navajowhite")
    else:
        status_text.set_text("RECOVERY — measurements restored")
        status_text.get_bbox_patch().set_facecolor("lightgreen")

    # Right panel
    sig_trail.set_data(t_2d[s], pos_sigma_drop[s])
    err_trail.set_data(t_2d[s], err_drop[s])
    time_marker.set_xdata([t_2d[k], t_2d[k]])

    return (meas_scat_d, true_trail_d, true_dot_d, pre_line, coast_line, post_line,
            est_dot_d, ell_drop, status_text, sig_trail, err_trail, time_marker)

anim_drop = FuncAnimation(fig, update_dropout, frames=len(frame_indices_d),
                          interval=60, blit=False)
plt.close(fig)

print(f"Animation: {len(frame_indices_d)} frames  |  Press ▶ to play")
display(HTML(anim_drop.to_jshtml()))

## Summary

In this notebook we built the Kalman filter from scratch — no estimation libraries, just matrices:

1. **Recursive Bayes structure** — the predict/update loop: motion model widens belief (more uncertain), measurement narrows it (more certain)
2. **1D constant-velocity tracking** — tracked position and velocity from noisy position-only measurements, watching the estimate converge
3. **Kalman gain convergence** — the gain starts large (trust measurements) and settles to steady state as the filter's confidence stabilizes
4. **Q/R tuning** — demonstrated stiff (smooth, slow), balanced, and floppy (noisy, responsive) filter personalities on a sinusoidal trajectory
5. **2D matrix-form filter** — generalized to a 4-state $[x, y, \dot{x}, \dot{y}]$ tracker with confidence ellipses
6. **Sensor dropout** — the filter coasts gracefully through a 60-step blackout, uncertainty grows honestly, and the Kalman gain spikes on recovery to snap the estimate back

### Key Takeaways

- The Kalman filter is **optimal** for linear-Gaussian systems — but only as good as its noise parameters ($Q$ and $R$) match reality
- **Predict widens, update narrows** — this rhythm is the heartbeat of all Bayesian estimation
- The Kalman gain is a **trust knob**: $K \approx 1$ means "follow the measurement", $K \approx 0$ means "trust the model"
- **Coasting** through measurement loss is not a failure mode — it's a designed feature. The filter knows it's becoming less certain and communicates that through growing $P$

### What's Next

- **N3** — Extended Kalman Filter (EKF): when the motion model is nonlinear, we linearize with Jacobians. Same predict/update structure, new math
- **N4** — Multi-Object Tracker: one Kalman filter per tracked object, with the Hungarian algorithm for data association — turning per-frame YOLO detections into persistent tracks

### References
- Kalman, R. E. (1960) — *A New Approach to Linear Filtering and Prediction Problems*
- Thrun, Burgard, & Fox — *Probabilistic Robotics*, Chapters 2–3 (Bayes filter and Kalman filter)
- Bar-Shalom, Li, & Kirubarajan — *Estimation with Applications to Tracking and Navigation*
- Welch & Bishop (2006) — *An Introduction to the Kalman Filter* (UNC Chapel Hill TR 95-041) — the classic accessible tutorial